# Big Data with PySpark — Small Labs 🔥

Today's session: three short, hands-on labs that teach the core Big Data patterns using **tiny datasets** (under 200 rows each), so every cell runs in seconds.

| Lab | Topic | Time |
|---|---|---|
| 1 | RDD Basics | ~10 min |
| 2 | Regression with MLlib | ~15 min |
| 3 | Classification with MLlib | ~15 min |

**Companion slides:** `BigData_Small_Labs.pptx`

> 💡 The code you write here is identical in shape to what you'd run on a real cluster with millions of rows — only the data size changes.

## 🔧 Setup — run this first

This installs Spark inside the Colab VM and starts a local Spark session using 2 simulated cores.

In [ ]:
!pip install pyspark --quiet

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("small-labs") \
    .master("local[2]") \
    .getOrCreate()

spark

---
# 🧪 Lab 1 — RDD Basics

**Goal:** understand *transformations* (lazy) vs. *actions* (run immediately) using a tiny list of products.

In [ ]:
products = ['laptop', 'mouse', 'keyboard', 'monitor',
            'webcam', 'charger', 'headset', 'mousepad']

rdd = spark.sparkContext.parallelize(products)
print(rdd)

Notice nothing about the *contents* printed above — `parallelize` just distributes the list into partitions. Let's try a transformation next.

In [ ]:
# Transformation — lazy, builds a plan but doesn't run yet
capped = rdd.map(lambda p: p.upper())
print(capped)          # still just an RDD object, not results

# Action — THIS is what actually runs it
print(capped.collect())

### ✍️ Your turn

Complete each cell below. Try to predict the output *before* running it.

In [ ]:
# 1. Use filter() to keep only products that contain the letter 'e'
products_with_e = rdd.filter(lambda p: 'e' in p)
# your code here


products_with_e.collect()

In [ ]:
# 2. Count how many products remain after the filter above
# your code here (use .count())


In [ ]:
# 3. Use take(3) to preview the first 3 original products WITHOUT collecting everything
# your code here


### 💬 Discussion
- Why did nothing print when we called `.map()` alone?
- What's the difference between `.take(3)` and `.collect()` on a huge (real) dataset?

---
# 🧪 Lab 2 — Regression with MLlib

**Goal:** predict a continuous number (exam score) from a small synthetic "study hours" dataset, using the same `fit()` / `transform()` pattern as any pyspark.ml model.

In [ ]:
import random
random.seed(42)

# small synthetic dataset: hours studied -> exam score (0-100)
rows = []
for _ in range(100):
    hours = round(random.uniform(0, 10), 1)
    score = min(100.0, max(0.0, 40 + hours * 6 + random.gauss(0, 8)))
    rows.append((hours, round(score, 1)))

df = spark.createDataFrame(rows, ["hours", "score"])
df.show(5)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

va = VectorAssembler(inputCols=["hours"], outputCol="features")
data = va.transform(df).select("features", "score")

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="score")
model = lr.fit(train)

predictions = model.transform(test)
predictions.show(5)

### ✍️ Your turn

In [ ]:
# 1. Print the model's rootMeanSquaredError and r2 from the training summary
summary = model.summary
# your code here



In [ ]:
# 2. Train a DecisionTreeRegressor on the same train/test split and compare its RMSE to the linear model
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# your code here




In [ ]:
# 3. Predict the exam score for a student who studies 7.5 hours
from pyspark.ml.linalg import Vectors

new_point = spark.createDataFrame([(Vectors.dense([7.5]),)], ["features"])
# your code here — call model.transform() on new_point and show it


### 💬 Discussion
- Which model had the lower RMSE — linear regression or the decision tree?
- What would you expect to happen to RMSE if we had 10,000 rows instead of 100?

---
# 🧪 Lab 3 — Classification with MLlib

**Goal:** predict a yes/no outcome (customer churn) from a small synthetic dataset, and evaluate it with ROC/AUC.

In [ ]:
import random
random.seed(7)

rows = []
for _ in range(120):
    tenure = random.randint(1, 72)          # months as a customer
    charges = round(random.uniform(20, 120), 2)  # monthly bill
    # churn is more likely with low tenure + high charges
    churn_score = (72 - tenure) * 0.02 + (charges - 60) * 0.015 + random.gauss(0, 1)
    label = 1 if churn_score > 1 else 0
    rows.append((tenure, charges, label))

df = spark.createDataFrame(rows, ["tenure_months", "monthly_charges", "label"])
df.show(5)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

va = VectorAssembler(inputCols=["tenure_months", "monthly_charges"], outputCol="features")
data = va.transform(df).select("features", "label")

train, test = data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

preds = model.transform(test)
preds.select("features", "label", "prediction", "probability").show(5, truncate=False)

In [ ]:
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator.evaluate(preds)
print(f"AUC: {auc:.3f}")

### ✍️ Your turn

In [ ]:
# 1. Is the AUC above closer to 0.5 (random guessing) or 1.0 (perfect)? Write a one-line comment with your read.
# your comment here


In [ ]:
# 2. Train a DecisionTreeClassifier on the same data and compare its AUC to logistic regression
from pyspark.ml.classification import DecisionTreeClassifier

# your code here




In [ ]:
# 3. Add a 3rd synthetic feature of your choice (e.g. 'num_support_calls'), rebuild the features vector,
#    retrain logistic regression, and see if AUC improves.
# your code here




### 💬 Discussion
- Which feature do you think matters more for predicting churn — tenure or monthly charges? How could you check?
- Why do we evaluate with AUC here instead of RMSE (which we used in Lab 2)?

---
# ✅ Wrap-up

You just ran the same **prepare → split → fit → transform → evaluate** pattern three times:

| Lab | Model(s) | Metric |
|---|---|---|
| RDD Basics | — | — |
| Regression | LinearRegression, DecisionTreeRegressor | RMSE, R² |
| Classification | LogisticRegression, DecisionTreeClassifier | AUC |

**Next steps**
- Save a copy of this notebook to your Drive: `File → Save a copy in Drive`
- Try pointing Lab 2 or Lab 3 at a real CSV (e.g. upload one and use `spark.read.csv(...)`) and watch the same code just work
- Revisit the *PySpark Deep Dive* quiz to check what stuck